# 02-04 정규화와 드롭아웃

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 레이어를 깊게 쌓으면 값의 크기가 흔들려요

In [ ]:
def stack_scale(n_layers=30, d=64, use_norm=False, gain=1.5):
    torch.manual_seed(0)
    x = torch.randn(256, d)  # (256, 64)
    stds = [x.std().item()]
    for _ in range(n_layers):
        layer = nn.Linear(d, d)
        nn.init.normal_(layer.weight, std=gain / d ** 0.5)
        x = torch.relu(layer(x))  # (256, 64)
        if use_norm:
            x = nn.functional.layer_norm(x, (d,))
        stds.append(x.std().item())
    return stds

for gain in [1.0, 2.0]:
    s = stack_scale(gain=gain)
    print(f"gain={gain}: 첫 레이어 표준편차 {s[1]:.3g}, 30번째 {s[-1]:.3g}")

In [ ]:
plt.figure(figsize=(6, 3.5))
for gain, color in [(1.0, "tab:blue"), (2.0, "tab:red")]:
    plt.plot(stack_scale(gain=gain), color=color, label=f"no norm, gain={gain}")
    plt.plot(stack_scale(gain=gain, use_norm=True), color=color, ls="--", label=f"LayerNorm, gain={gain}")
plt.yscale("log")
plt.xlabel("layer")
plt.ylabel("std of activations")
plt.legend()
plt.show()

## 2. 레이어 정규화(LayerNorm)

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0], [10.0, 20.0, 30.0, 40.0]])  # (2, 4)
mean = x.mean(dim=-1, keepdim=True)  # (2, 1)
var = x.var(dim=-1, keepdim=True, unbiased=False)  # (2, 1)
manual = (x - mean) / torch.sqrt(var + 1e-5)  # (2, 4)
print(manual)

ln = nn.LayerNorm(4)
print(ln(x))
print(ln.weight, ln.bias)  # 학습할 수 있는 배율과 이동값, 처음에는 1과 0

In [ ]:
torch.manual_seed(0)
x = torch.randn(3, 64) * torch.tensor([[0.5], [3.0], [10.0]]) + torch.tensor([[5.0], [-2.0], [20.0]])  # (3, 64)
y = nn.LayerNorm(64)(x).detach()  # (3, 64)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
for i, color in enumerate(["tab:blue", "tab:green", "tab:purple"]):
    axes[0].hist(x[i], bins=20, alpha=0.6, color=color, label=f"row {i}")
    axes[1].hist(y[i], bins=20, alpha=0.6, color=color, label=f"row {i}")
axes[0].set_title("before LayerNorm")
axes[1].set_title("after LayerNorm")
axes[0].legend()
plt.show()

## 3. 드롭아웃

In [ ]:
torch.manual_seed(0)
drop = nn.Dropout(p=0.5)
x = torch.ones(2, 8)  # (2, 8)

drop.train()  # 학습 모드
print(drop(x))
drop.eval()  # 평가 모드
print(drop(x))

In [ ]:
torch.manual_seed(0)
drop.train()
masks = drop(torch.ones(4, 32))  # (4, 32)
plt.figure(figsize=(8, 1.8))
plt.imshow(masks, cmap="Greys_r")
plt.title("Dropout(p=0.5) in train mode: black = dropped, white = kept x2")
plt.axis("off")
plt.show()

In [ ]:
drop.train()
x = torch.ones(10000)
print("평균(학습 모드):", drop(x).mean().item())
print("0이 된 비율:", (drop(x) == 0).float().mean().item())

## 4. model.train()과 model.eval()

In [ ]:
model = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5), nn.LayerNorm(4))
x = torch.ones(1, 4)
model.train()
print(model(x))
print(model(x))  # 매번 다르게 꺼져요
model.eval()
print(model(x))
print(model(x))  # 항상 같아요